# Circular resistive-wall pipe — TD Legacy vs. Conformal FIT

**Goal:** Compare the **volumetric conductivity** treatment in WAKIS for `geometry_mode='legacy'` and `'conformal'`, each with (A) PEC on all boundaries and (B) PEC in x/y + PML or CPML in z. The **FD SIBC comparison is intentionally deferred**.

The geometry is a straight circular vacuum channel inside a thick conducting cylinder, itself surrounded by PEC background. Two overlapping watertight STL volumes are used, **outer conductor first, inner vacuum second**. Their capped ends are extended *beyond* the computational z domain, so the beam channel is open up to the boundary planes. No TD SIBC is enabled: `use_sibc=False` and `stl_material_types={'shell':'normal','vacuum':'normal'}`.

**Important caveats:**
- The source crosses artificial PEC end planes in the all-PEC variant; this is an approximate finite-domain impedance benchmark, not a literal open infinite beam pipe. `skip_cells` removes end-adjacent cells from wake integration, not from EM evolution.
- `PML/CPML` only changes axial boundaries. The solver may switch the source to `tfsf` for these BCs, so compare boundary effects **and** source implementation; not automatically an apples-to-apples difference.
- At σ=300 S/m the explicit solver's conductivity-based relaxation time (`eps0/sigma`) is very short. The notebook estimates steps and guards expensive runs.
- A straight infinitely long pipe has a different mathematical problem from the finite simulation; analytical impedance is a benchmark, not a guarantee of exact agreement.


In [ ]:
from pathlib import Path
from contextlib import contextmanager
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyvista as pv
from scipy.constants import c, epsilon_0, mu_0, pi
from wakis import GridFIT3D, SolverFIT3D, WakeSolver

plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": 0.25})
print("Imported WAKIS:", getattr(__import__("wakis"), "__file__", "unknown"))

## 1. Parameters — change them here only

In [ ]:
# ----------------------------- Geometry -----------------------------
PIPE_RADIUS = 0.020  # m: vacuum bore radius
WALL_THICKNESS = 0.012  # m: conductive radial thickness
PIPE_LENGTH = 0.300  # m: length inside computational domain
XY_PADDING = 0.006  # m: PEC background outside metal
STL_AXIAL_EXTENSION = 0.040  # m beyond each computational end
STL_RESOLUTION = 192  # polygonization of the circular walls
SIGMA = 300.0  # S/m, VOLUMETRIC in both TD variants
EPS_R = MU_R = 1.0

# ------------------------------- Mesh -------------------------------
NX = NY = 48  # quick geometry / coarse initial TD test; refine later
NZ = 64  # independent longitudinal resolution
AREA_FLOOR = 0.1  # conformal area floor (minimum cut-face area fraction)
STL_METHOD = "interior_points"

# ---------------------------- Beam + wake ---------------------------
Q_BUNCH = 1e-9
BETA = 1.0
SIGMA_Z = 0.030  # m, Gaussian bunch length
WAKELENGTH = 0.60  # m, also determines spectral resolution
SKIP_CELLS = 5
F_MIN = 0.10e9
F_MAX = 1.00e9
N_FREQUENCY = 301

# ------------------------- Boundary conditions ----------------------
# Choose 'PML' or 'CPML' according to your current WAKIS branch.
AXIAL_ABSORBER = "PML"
N_PML = 8
CASES = {
    "legacy_pec": ("legacy", "PEC"),
    "conformal_pec": ("conformal", "PEC"),
    "legacy_pml": ("legacy", AXIAL_ABSORBER),
    "conformal_pml": ("conformal", AXIAL_ABSORBER),
}

# Guard: to launch all runs, explicitly enable both parameters after checking preview.
RUN_SIMULATIONS = False
ALLOW_EXPENSIVE_RUNS = False
MAX_TIMESTEPS = 30_000
REUSE_RESULTS = True

OUTDIR = (Path.cwd() / "circular_pipe_rw_td_results").resolve()
GEOM_DIR = OUTDIR / "geometry"
OUTDIR.mkdir(parents=True, exist_ok=True)
GEOM_DIR.mkdir(parents=True, exist_ok=True)
R_OUT = PIPE_RADIUS + WALL_THICKNESS
DOMAIN_HALF_XY = R_OUT + XY_PADDING
XMIN = YMIN = -DOMAIN_HALF_XY
XMAX = YMAX = DOMAIN_HALF_XY
ZMIN, ZMAX = -PIPE_LENGTH / 2, PIPE_LENGTH / 2
print("Output:", OUTDIR)
print("Geometry: bore radius", PIPE_RADIUS, "m, outer radius", R_OUT, "m")

## 2. Analytical estimates and a realistic runtime warning

In [ ]:
f_cutoff = 1.84118378134 * c / (2 * pi * PIPE_RADIUS)
frequencies = np.linspace(F_MIN, F_MAX, N_FREQUENCY)
skin_depth = np.sqrt(1 / (pi * frequencies * mu_0 * SIGMA))
Rs = np.sqrt(pi * frequencies * mu_0 / SIGMA)

# Standard good-conductor, thick-wall asymptotic impedance per unit length.
# For the e^{+j omega t} convention: Zs=(1+j)Rs.
ZL_analytic = (1 + 1j) * Rs / (2 * pi * PIPE_RADIUS)

dx, dy, dz = ((XMAX - XMIN) / NX, (YMAX - YMIN) / NY, PIPE_LENGTH / NZ)
dt_cfl = 0.5 / (c * np.sqrt(dx**-2 + dy**-2 + dz**-2))
dt_relax = epsilon_0 * EPS_R / SIGMA
dt_est = min(dt_cfl, dt_relax)
# WakeSolver uses an injection offset; default ti approximately 8.53 sigma_z / v.
ti_est = 8.53 * SIGMA_Z / (BETA * c)
tmax_est = ti_est + (WAKELENGTH + PIPE_LENGTH) / (BETA * c)
steps_est = int(np.ceil(tmax_est / dt_est))
print(f"TE11 cutoff = {f_cutoff / 1e9:.3f} GHz")
print(f"dx={dx * 1e3:.3f} mm; dz={dz * 1e3:.3f} mm")
print(f"δ(0.3 GHz)={np.sqrt(1 / (pi * 0.3e9 * mu_0 * SIGMA)) * 1e3:.3f} mm")
print(f"δ(1 GHz)  ={np.sqrt(1 / (pi * 1e9 * mu_0 * SIGMA)) * 1e3:.3f} mm")
print(f"CFL dt≈{dt_cfl:.3e} s, conductivity dt≈{dt_relax:.3e} s")
print(f"Estimated dt≈{dt_est:.3e} s; simulation ~{steps_est:,} time steps per case")
if steps_est > MAX_TIMESTEPS:
    print(
        "WARNING: expensive explicit-conductivity TD simulation! "
        "Set ALLOW_EXPENSIVE_RUNS=True only after checking the solver timestep."
    )

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(frequencies / 1e9, skin_depth * 1e3)
axes[0].axhline(dx * 1e3, linestyle="--", color="C1", label="Transverse grid step")
axes[0].set(
    xlabel="Frequency [GHz]", ylabel="Skin depth [mm]", title="Skin depth resolution"
)
axes[0].legend()
axes[1].plot(frequencies / 1e9, ZL_analytic.real, label="Re")
axes[1].plot(frequencies / 1e9, ZL_analytic.imag, "--", label="Im")
axes[1].set(
    xlabel="Frequency [GHz]",
    ylabel="Analytic Z∥ / L [Ω/m]",
    title="Thick-wall approximation",
)
axes[1].legend()
plt.tight_layout()
plt.show()

## 3. Create two closed, overlapping STL cylinders (open pipe within grid)

The cylinders are watertight and extend *beyond* the simulation box in z. The inner vacuum volume overrides the outer conductive volume in the overlap. For legacy, STL order controls material overwrites; for conformal, it controls the point-region precedence.

In [ ]:
SHELL_STL = GEOM_DIR / "pipe_outer_conductor_volume.stl"
VACUUM_STL = GEOM_DIR / "pipe_inner_vacuum_volume.stl"

# Both cylinder caps outside the computational domain.
stl_height = PIPE_LENGTH + 2 * STL_AXIAL_EXTENSION


def closed_cylinder(radius):
    return (
        pv.Cylinder(
            center=(0, 0, 0),
            direction=(0, 0, 1),
            radius=radius,
            height=stl_height,
            resolution=STL_RESOLUTION,
            capping=True,
        )
        .triangulate()
        .clean()
    )


closed_cylinder(R_OUT).save(str(SHELL_STL))
closed_cylinder(PIPE_RADIUS).save(str(VACUUM_STL))

shell_mesh = pv.read(str(SHELL_STL))
vacuum_mesh = pv.read(str(VACUUM_STL))
assert shell_mesh.n_open_edges == 0, "Outer STL is not closed"
assert vacuum_mesh.n_open_edges == 0, "Inner STL is not closed"
assert float(np.max(np.abs(vacuum_mesh.bounds[4:]))) >= PIPE_LENGTH / 2
print("Shell STL:", SHELL_STL)
print("Vacuum STL:", VACUUM_STL)
print("Outer bounds:", shell_mesh.bounds)
print("Inner bounds:", vacuum_mesh.bounds)

# Optional 3D display (requires notebook vtk/pyvista backend)
try:
    pv.set_jupyter_backend("static")
    p = pv.Plotter(notebook=True)
    p.add_mesh(shell_mesh, color="gray", opacity=0.18)
    p.add_mesh(vacuum_mesh, color="deepskyblue", opacity=0.45)
    p.show_grid()
    p.show()
except Exception as exc:
    print("3D preview unavailable:", exc)

## 4. Construct matched grids and TD solvers

The material assignments are **volumetric normal materials** in *both* TD variants. The external background is PEC. No TD SIBC approximation is requested.

In [ ]:
STL_SOLIDS = {"shell": str(SHELL_STL), "vacuum": str(VACUUM_STL)}
STL_MATERIALS = {
    "shell": [EPS_R, MU_R, SIGMA],
    "vacuum": [1.0, 1.0, 0.0],
}
STL_MATERIAL_TYPES = {"shell": "normal", "vacuum": "normal"}


def make_grid(mode):
    return GridFIT3D(
        xmin=XMIN,
        xmax=XMAX,
        ymin=YMIN,
        ymax=YMAX,
        zmin=ZMIN,
        zmax=ZMAX,
        Nx=NX,
        Ny=NY,
        Nz=NZ,
        stl_solids=STL_SOLIDS,
        stl_materials={k: list(v) for k, v in STL_MATERIALS.items()},
        stl_material_types=STL_MATERIAL_TYPES.copy(),
        geometry_mode=mode,
        stl_method=STL_METHOD,
        bg_material_type="pec",
        conformal_stabilization="area_floor" if mode == "conformal" else None,
        conformal_stabilization_fraction=AREA_FLOOR,
        stl_tol=1e-3,
        verbose=0,
    )


def make_wake(case_dir):
    return WakeSolver(
        q=Q_BUNCH,
        sigmaz=SIGMA_Z,
        beta=BETA,
        xsource=0.0,
        ysource=0.0,
        xtest=0.0,
        ytest=0.0,
        skip_cells=SKIP_CELLS,
        save=False,
        results_folder=str(case_dir),
        verbose=0,
    )


def make_td_solver(grid, wake, axial_bc):
    low = ["PEC", "PEC", axial_bc]
    high = ["PEC", "PEC", axial_bc]
    return SolverFIT3D(
        grid=grid,
        wake=wake,
        bc_low=low,
        bc_high=high,
        use_stl=True,
        use_sibc=False,
        bg="pec",
        cfln=0.5,
        n_pml=N_PML,
        fmax=F_MAX,
        dtype=np.float64,
        verbose=0,
    )


@contextmanager
def in_directory(folder):
    folder = Path(folder).resolve()
    folder.mkdir(parents=True, exist_ok=True)
    previous = Path.cwd()
    try:
        os.chdir(folder)
        yield
    finally:
        os.chdir(previous)

## 5. Inspect material masks before solving

The checks below compare the raw geometrical point masks, the *effective* volumetric conductivity tensor on a central z slice, and the interior vacuum region. This is especially important because the legacy and conformal material assignment differ.

In [ ]:
grid_preview = {}
for mode in ("legacy", "conformal"):
    grid_preview[mode] = make_grid(mode)
    grid = grid_preview[mode]
    print(f"\n{mode}: Nx×Ny×Nz = {grid.Nx}×{grid.Ny}×{grid.Nz}")
    for key in STL_SOLIDS:
        mask = np.asarray(grid.grid[key], dtype=bool)
        print(f"   STL {key}: {mask.sum()} occupied cells")
    if mode == "conformal":
        for rid, key in grid.region_id_to_key.items():
            print(
                "   Region",
                rid,
                key,
                ":",
                int(np.count_nonzero(grid.region_ids["primal"] == rid)),
                "primal points",
            )

# Cross section: note both raw STL masks overlap in vacuum; the later 'vacuum' overrides shell.
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (mode, grid) in zip(axes, grid_preview.items()):
    shell_cell = np.asarray(grid.grid["shell"], bool).reshape(NX, NY, NZ)
    vac_cell = np.asarray(grid.grid["vacuum"], bool).reshape(NX, NY, NZ)
    assigned = np.zeros((NX, NY), dtype=int)
    assigned[shell_cell[:, :, NZ // 2]] = 1
    assigned[vac_cell[:, :, NZ // 2]] = 2
    im = ax.pcolormesh(
        grid.x[:-1] * 1e3,
        grid.y[:-1] * 1e3,
        assigned.T,
        shading="nearest",
        cmap="viridis",
        vmin=0,
        vmax=2,
    )
    for radius in (PIPE_RADIUS, R_OUT):
        ax.add_patch(
            plt.Circle((0, 0), radius * 1e3, fill=False, color="white", ls="--", lw=1.3)
        )
    ax.set(title=mode, xlabel="x [mm]", ylabel="y [mm]", aspect="equal")
plt.suptitle("Cell-mask overview: 0=PEC, 1=conductor, 2=vacuum")
plt.tight_layout()
plt.show()

## 6. Solver-level conductivity sanity check and actual step count

The solver may use a conductivity time-step restriction considerably stricter than CFL. Only one inexpensive solver initialization per geometry mode is done here; it does **not** run the time stepping. Inspect `sigma` on a central cross section to verify that the conductive shell is not accidentally modeled as PEC or vacuum.

In [ ]:
solver_preview = {}
preview_rows = []
for mode in ("legacy", "conformal"):
    d = OUTDIR / ("preview_" + mode)
    w = make_wake(d)
    s = make_td_solver(grid_preview[mode], w, "PEC")
    solver_preview[mode] = s
    sig = np.asarray(s.sigma.field_x, float)
    nonzero = sig > 1e-12
    preview_rows.append(
        {
            "mode": mode,
            "dt_s": float(s.dt),
            "expected_steps": int(np.ceil(tmax_est / float(s.dt))),
            "sigma_nonzero_fraction": float(np.mean(nonzero)),
            "sigma_min_nonzero": float(sig[nonzero].min()) if nonzero.any() else np.nan,
            "sigma_max": float(sig.max()),
            "use_conductivity": bool(s.use_conductivity),
        }
    )
preview_df = pd.DataFrame(preview_rows)
display(preview_df)
assert all(preview_df.use_conductivity), "Volumetric conductivity was not enabled"

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for ax, mode in zip(axs, ("legacy", "conformal")):
    s = solver_preview[mode]
    sig = np.asarray(s.sigma.field_x, float)
    # Fields generally store Nx×Ny×Nz for FIT x/y/z arrays.
    if sig.shape == (NX, NY, NZ):
        a = sig[:, :, NZ // 2]
        ax.pcolormesh(s.grid.x[:-1] * 1e3, s.grid.y[:-1] * 1e3, a.T, shading="nearest")
    else:
        ax.text(
            0.05,
            0.5,
            f"Unexpected sigma.field_x shape: {sig.shape}",
            transform=ax.transAxes,
        )
    ax.set(
        title=f"{mode}: sigma_x [S/m]", xlabel="x [mm]", ylabel="y [mm]", aspect="equal"
    )
plt.tight_layout()
plt.show()

max_actual_steps = int(preview_df.expected_steps.max())
print(f"Max estimated actual steps per case: {max_actual_steps:,}")

## 7. Run the four cases (explicit opt-in)

**Set `RUN_SIMULATIONS=True`** in the parameter cell, and if the conductivity timestep makes the run expensive, also set `ALLOW_EXPENSIVE_RUNS=True`. Each run has its own working folder; WAKIS-generated `Ez.h5` and other artifacts therefore cannot overwrite another case. Results are also saved as small `.npz` files and can be plotted later without rerunning simulations.

`skip_cells` is set in `WakeSolver`, as supported by the current API; `add_space` is intentionally not used.

In [ ]:
def result_file(case_name):
    return OUTDIR / case_name / "wake_results.npz"


def run_case(case_name, mode, axial_bc):
    case_dir = OUTDIR / case_name
    case_dir.mkdir(parents=True, exist_ok=True)
    path = result_file(case_name)
    if REUSE_RESULTS and path.exists():
        print("Reusing:", path)
        return path
    grid = grid_preview[mode]
    wake = make_wake(case_dir)
    solver = make_td_solver(grid, wake, axial_bc)
    nt_est = int(np.ceil(tmax_est / float(solver.dt)))
    print(
        f"[{case_name}] dt={float(solver.dt):.3e} s; ≈{nt_est:,} time steps; "
        f"source_type={solver.source_type}"
    )
    if nt_est > MAX_TIMESTEPS and not ALLOW_EXPENSIVE_RUNS:
        raise RuntimeError(
            f"{case_name}: estimated {nt_est:,} time steps exceeds MAX_TIMESTEPS "
            f"({MAX_TIMESTEPS:,}). Set ALLOW_EXPENSIVE_RUNS=True explicitly."
        )

    tic = time.perf_counter()
    with in_directory(case_dir):
        # Uses the development branch's TD wakefield pipeline.
        solver.wakesolve(
            wakelength=WAKELENGTH,
            wake=wake,
            plot=False,
            save_J=False,
        )
    runtime = time.perf_counter() - tic
    # Current WakeSolver stores the longitudinal impedance and wake here.
    wake.calc_long_Z(samples=4001, fmax=F_MAX)
    f = np.asarray(wake.f, float).reshape(-1)
    Z = np.asarray(wake.Z, complex).reshape(-1)
    s = np.asarray(wake.s, float).reshape(-1)
    wp = np.asarray(wake.WP, float).reshape(-1)
    assert len(f) == len(Z) and len(f) > 2, "Unexpected frequency-domain wake output"
    np.savez_compressed(
        path,
        f_hz=f,
        Z_ohm=Z,
        s_m=s,
        WP=wp,
        case=case_name,
        mode=mode,
        axial_bc=axial_bc,
        sigma=SIGMA,
        pipe_length=PIPE_LENGTH,
        pipe_radius=PIPE_RADIUS,
        dt_s=float(solver.dt),
        estimated_steps=nt_est,
        runtime_s=runtime,
        skip_cells=SKIP_CELLS,
        source_type=str(solver.source_type),
    )
    print("Saved:", path, f"({runtime:.1f} s)")
    return path


if RUN_SIMULATIONS:
    if max_actual_steps > MAX_TIMESTEPS and not ALLOW_EXPENSIVE_RUNS:
        raise RuntimeError(
            "Explicit conductivity timestep is expensive. "
            "Set ALLOW_EXPENSIVE_RUNS=True if intentional."
        )
    for name, (mode, bc) in CASES.items():
        run_case(name, mode, bc)
else:
    print("RUN_SIMULATIONS=False: preview completed; no time stepping performed.")

## 8. Load saved cases and compare wake potentials

In [ ]:
results = {}
for name in CASES:
    path = result_file(name)
    if path.exists():
        with np.load(path, allow_pickle=False) as arr:
            results[name] = {key: arr[key] for key in arr.files}
print("Available results:", list(results))

if results:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for name, res in results.items():
        ax.plot(res["s_m"], res["WP"], label=name)
    ax.set(
        xlabel="s [m]",
        ylabel="Longitudinal wake potential [V/pC]",
        title="Wake potential",
    )
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print(
        "Run the time-domain cases first or place prior wake_results.npz files in each case folder."
    )

## 9. Longitudinal resistive-wall impedance: analytical vs. WAKIS

WAKIS reports total impedance [Ω] for its finite integration region. Here we divide by the nominal pipe length. Since `skip_cells` reduces the integrated region, also compare with the approximate effective length `PIPE_LENGTH - 2*SKIP_CELLS*dz`. The latter is only an estimate; verify the exact WAKIS integration range for a precise normalization.

In [ ]:
if results:
    fig, axs = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True)
    for name, res in results.items():
        f = np.asarray(res["f_hz"])
        z = np.asarray(res["Z_ohm"]) / PIPE_LENGTH
        band = (f >= F_MIN) & (f <= F_MAX)
        axs[0].plot(f[band] / 1e9, z.real[band], label=name)
        axs[1].plot(f[band] / 1e9, z.imag[band], label=name)
    axs[0].plot(
        frequencies / 1e9, ZL_analytic.real, "k--", lw=2, label="Analytical thick-wall"
    )
    axs[1].plot(
        frequencies / 1e9, ZL_analytic.imag, "k--", lw=2, label="Analytical thick-wall"
    )
    for ax, title in zip(axs, ("Real part", "Imaginary part")):
        ax.set(xlabel="Frequency [GHz]", ylabel="Z∥ / L [Ω/m]", title=title)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # Explicit normalization sensitivity from skip_cells.
    Leff = PIPE_LENGTH - 2 * SKIP_CELLS * dz
    print(
        f"Nominal L = {PIPE_LENGTH:.4f} m; approximate integrated L_eff = {Leff:.4f} m"
    )
else:
    print("No calculated cases available.")

## 10. Compare boundary conditions and discretizations separately

In [ ]:
def compare_pair(name_a, name_b, title):
    if name_a not in results or name_b not in results:
        print("Missing:", name_a, "or", name_b)
        return
    a = results[name_a]
    b = results[name_b]
    f = np.linspace(F_MIN, F_MAX, 301)

    def interp_complex(r):
        x = np.asarray(r["f_hz"])
        y = np.asarray(r["Z_ohm"]) / PIPE_LENGTH
        ii = np.argsort(x)
        return np.interp(f, x[ii], y.real[ii]) + 1j * np.interp(f, x[ii], y.imag[ii])

    za, zb = interp_complex(a), interp_complex(b)
    fig, axs = plt.subplots(1, 2, figsize=(12, 3.9))
    axs[0].plot(f / 1e9, za.real, label=name_a)
    axs[0].plot(f / 1e9, zb.real, label=name_b)
    axs[0].set(ylabel="Re(Z∥/L) [Ω/m]", xlabel="Frequency [GHz]")
    axs[1].plot(f / 1e9, za.imag, label=name_a)
    axs[1].plot(f / 1e9, zb.imag, label=name_b)
    axs[1].set(ylabel="Im(Z∥/L) [Ω/m]", xlabel="Frequency [GHz]")
    fig.suptitle(title)
    for ax in axs:
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


compare_pair("legacy_pec", "conformal_pec", "Discretization comparison: PEC ends")
compare_pair("legacy_pml", "conformal_pml", "Discretization comparison: absorbing ends")
compare_pair("legacy_pec", "legacy_pml", "Boundary comparison: legacy")
compare_pair("conformal_pec", "conformal_pml", "Boundary comparison: conformal")

## 11. Follow-up convergence checklist

1. Verify the **actual axial source and wake integration range** for PEC and PML. Check `source_type` stored in the NPZ; a PML run may automatically use TFSF injection.
2. Run a **mesh refinement sweep** in the transverse directions until there are multiple cells per skin depth in the frequency band of interest. Re-run the geometry preview each time.
3. Verify that the shell conductivity tensor remains finite and nonzero in both modes, especially around the curved inner wall.
4. Change **`SKIP_CELLS`** (recompute wake impedance from saved `Ez.h5` if supported) and the pipe length to estimate end effects.
5. Once TD is understood, add a **direct FD SIBC** calculation on the same inner boundary and compare with the analytical reference.

**Note:** Using 300 S/m intentionally tests bulk losses, but at this conductivity the explicit conductivity timestep can dominate performance even with a coarse mesh. If runtime becomes prohibitive, an appropriate dissipative time integrator (if available and validated in this branch) or a lower conductivity is preferable to silently bypassing the stability constraint.